# 实验三：YOLO 后处理与 CPU NMS 基线

YOLO 模型输出通常还不是最终检测框。端侧部署中，后处理可能成为 CPU 侧瓶颈。本章用 NumPy 实现 CPU NMS，建立后续 NPU 卸载优化的基线。

## YOLOv5s 的输出基线

本章后处理基线：模型为 YOLOv5s，输入为 `1×3×640×640`，类别为 COCO 80 类，raw output 按 `1×25200×85` 解析。其中 85 = 4 个框坐标 + 1 个 objectness + 80 个类别分数。

这里讨论的是端侧推理后的后处理，不需要训练集；用于验证的图片来自 `src/data/images` 中的少量 COCO 风格样例图。


## 后处理阶段

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">阶段</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">decode</td>
      <td style="text-align: left;">将模型 raw output 解码为候选框、objectness、类别分数</td>
    </tr>
    <tr>
      <td style="text-align: left;">score filter</td>
      <td style="text-align: left;">删除低置信度候选框，减少 NMS 输入数量</td>
    </tr>
    <tr>
      <td style="text-align: left;">NMS</td>
      <td style="text-align: left;">抑制高度重叠的框，保留高分框</td>
    </tr>
    <tr>
      <td style="text-align: left;">坐标映射</td>
      <td style="text-align: left;">将 640×640 letterbox 输入尺度映射回原图坐标</td>
    </tr>
    <tr>
      <td style="text-align: left;">可视化/上报</td>
      <td style="text-align: left;">绘框、保存结果或发送到业务系统</td>
    </tr>
  </tbody>
</table>

In [ ]:
# ====== 1. 运行 CPU 后处理基线 ======
from src.scripts.pyacl_yolo_infer import load_config, preprocess, PyAclYoloSession
from src.scripts.benchmark_postprocess import postprocess_cpu, timeit
from pathlib import Path

cfg = load_config('src/configs/yolo_edge.yaml')
input_tensor = preprocess('src/data/images/bus.jpg', cfg)

with PyAclYoloSession(cfg) as session:
    pred_real = session.infer(input_tensor)

detections = postprocess_cpu(pred_real, cfg)

names = Path(cfg["data"]["class_names"]).read_text(encoding="utf-8").splitlines()

print("detections:", len(detections))
for x1, y1, x2, y2, score, cls_id in detections[:20]:
    cls_id = int(cls_id)
    print(f"{names[cls_id]:12s} score={score:.3f} box=({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})")

In [ ]:
# ====== 2. 重复计时，得到 CPU NMS 延迟 ======
from src.scripts.benchmark_postprocess import timeit

cpu_ms, result = timeit(lambda: postprocess_cpu(pred_real, cfg), repeat=30, warmup=5)
print(f"CPU postprocess latency: {cpu_ms:.3f} ms")
print(f"detections: {len(result)}")

## 为什么要卸载后处理

在端侧应用中，模型推理已经在 NPU 上执行，但后处理如果留在 CPU，可能出现：

- NPU 等待 CPU 后处理结束，端到端 FPS 上不去。
- 候选框多时 NMS 复杂度上升，CPU 延迟波动明显。
- 多路视频输入时，CPU 同时承担解码、预处理、业务逻辑，后处理容易成为热点。

因此本实验把 NMS 作为异构卸载的代表任务。

In [ ]:
# ====== 3. 命令行基准测试 ======
print('python src/scripts/benchmark_postprocess.py --config src/configs/yolo_edge.yaml --repeat 100')

---
## 本章小结

CPU NMS 基线已经建立。下一章将设计自定义后处理算子，并验证可用性。

## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) YOLOv5s 输出 `[N,85]` 中前 4 个数通常表示什么？
   - A. xywh 边框信息
   - B. 类别名称字符串
   - C. NPU 内存地址
   - D. 图片路径

2. (单选题) CPU NMS baseline 的主要作用是？
   - A. 作为自定义 NPU NMS 的正确性参考
   - B. 替代 ATC 转换
   - C. 删除 OM 模型
   - D. 安装 CANN

3. (单选题) 后处理中过滤候选框时，常用的分数计算是？
   - A. objectness * class score
   - B. x_center + y_center
   - C. 图片宽度 * 高度
   - D. NPU 温度 / 功耗

4. (单选题) NMS 的核心目的是？
   - A. 抑制高度重叠的冗余框
   - B. 把图片转成 NCHW
   - C. 把 ONNX 转成 OM
   - D. 启动 Jupyter

5. (多选题) YOLO 后处理通常包含哪些步骤？
   - A. 坐标从 xywh 转 xyxy
   - B. 按 score 阈值过滤
   - C. 按类别或全局执行 NMS
   - D. 保留最终检测框和类别

6. (多选题) CPU baseline 中需要保存或观察哪些信息，方便后续 NPU 对齐？
   - A. boxes
   - B. scores
   - C. keep 索引
   - D. count 数量

7. (多选题) 影响 NMS 输出的参数包括哪些？
   - A. score_threshold
   - B. nms_iou_threshold
   - C. max_detections
   - D. class_agnostic

8. (判断题) CPU NMS baseline 的意义只在于测速，不需要用于正确性对齐。

9. (判断题) 如果输入候选框已经按 score 降序排列，NMS kernel 内部可以暂时不实现排序。

10. (填空题) NMS 中判断两个框重叠程度的常用指标是 `____`。

11. (填空题) 本实验默认的 NMS IoU 阈值是 `____`。

12. (简答题) 为什么在开发自定义算子前必须先有 CPU baseline？

13. (简答题) class_agnostic=false 时，NMS 与 class_agnostic=true 有什么区别？

14. (简答题) 为什么真实 YOLO 输出过滤后候选框数量可能远大于最初小样例的 51？

15. (代码设计题) 写一段代码，计算 YOLO 候选框最终 scores 并按阈值过滤。

> 参考答案见 answer/04.04_yolo_postprocess_cpu_nms_baseline_answer.ipynb。
